# 1. Ranked Actions + Reason Codes

The model produces a ranked list of content pages that should be reviewed first. Higher-ranked pages are expected to have a greater likelihood of requiring content updates based on observed search and engagement signals.

| Priority | Reason Code | Recommended Action |
|----------|-------------|-------------------|
| High | Declining trend + Low CTR | Review and refresh content immediately |
| High | Declining trend + Poor average position | Improve SEO and update content |
| Medium | Low engagement | Improve readability and content quality |
| Medium | Older content | Review for outdated information |
| Low | Stable performance | Continue monitoring |

In [5]:
import os
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
try:
    df
except NameError:
    from google.colab import files

    uploaded = files.upload()
    filename = next(iter(uploaded))
    df = pd.read_csv(filename)

data = df.copy()

# Proxy target used in the starter exercise
data["target"] = (data["trend_direction"] == "down").astype(int)

# Exclude trend_direction and trend_pct because they define the target
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

features = [column for column in features if column in data.columns]

X = data[features]
y = data["target"]

# Missing indicators avoid silently treating missing values as genuine zeros
queue_model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=8,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

# Validation was performed separately in W06.
# This full-data fit is only for producing the review queue.
queue_model.fit(X, y)

data["review_probability"] = queue_model.predict_proba(X)[:, 1]
data["priority_score"] = (100 * data["review_probability"]).round(2)


def create_reason_codes(row):
    reasons = []

    if row.get("impressions_90d", 0) >= 500:
        reasons.append("meaningful_visibility")

    if (
        row.get("impressions_90d", 0) >= 500
        and 0 < row.get("avg_position", 0) <= 20
        and row.get("ctr", np.inf) < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row.get("days_since_last_update", 0) >= 180
        and row.get("impressions_90d", 0) >= 500
    ):
        reasons.append("stale_visible_page")

    if (
        row.get("content_age_days", 0) >= 180
        and 0 < row.get("avg_position", 0) <= 10
    ):
        reasons.append("page_one_decay_risk")

    if (
        row.get("word_count", np.inf) < 1200
        and row.get("word_count", 0) > 0
        and row.get("impressions_90d", 0) >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        row.get("sessions_90d", 0) >= 30
        and (
            row.get("engagement_rate", np.inf) < 30
            or row.get("scroll_rate", np.inf) < 30
        )
    ):
        reasons.append("low_engagement")

    if not reasons:
        reasons.append("model_review_signal")

    return "|".join(reasons)


def recommend_action(reason_codes):
    reasons = set(reason_codes.split("|"))

    if "low_ctr_visible_page" in reasons:
        return "Review title, metadata, intent match, and search snippet"

    if "page_one_decay_risk" in reasons or "stale_visible_page" in reasons:
        return "Review for refresh, outdated information, and missing sections"

    if "thin_visible_page" in reasons:
        return "Assess whether useful expansion is justified"

    if "low_engagement" in reasons:
        return "Review structure, readability, scanability, and intent fulfilment"

    return "Manually inspect and decide whether to refresh or monitor"


data["reason_codes"] = data.apply(create_reason_codes, axis=1)
data["recommended_action"] = data["reason_codes"].apply(recommend_action)

data["priority_band"] = pd.cut(
    data["priority_score"],
    bins=[-1, 50, 70, 100],
    labels=["Monitor", "Medium", "High"]
)

queue_columns = [
    "content_id",
    "client_id",
    "priority_score",
    "priority_band",
    "reason_codes",
    "recommended_action",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "scroll_rate"
]

queue_columns = [column for column in queue_columns if column in data.columns]

ranked_queue = (
    data[queue_columns]
    .sort_values("priority_score", ascending=False)
    .reset_index(drop=True)
)

display(ranked_queue.head(20))

print("\nPriority distribution:")
print(ranked_queue["priority_band"].value_counts())

print("\nTop reason codes:")
print(ranked_queue["reason_codes"].value_counts().head(10))

,content_id,client_id,priority_score,priority_band,reason_codes,recommended_action,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,engagement_rate,scroll_rate
0,content_65bf969d1247,client_3fdba35f04,79.78,High,meaningful_visibility|low_ctr_visible_page,"Review title, metadata, intent match, and sear...",1092,1,0.09,12.0,139,104,7.69,23.08
1,content_9b6df29f7889,client_3fdba35f04,79.59,High,meaningful_visibility|low_ctr_visible_page,"Review title, metadata, intent match, and sear...",1622,2,0.12,3.1,139,104,5.00,13.64
2,content_1e446b05f1c5,client_3fdba35f04,79.58,High,meaningful_visibility|low_engagement,"Review structure, readability, scanability, an...",685,0,0.00,34.0,165,104,0.00,27.08
3,content_04d7435934f0,client_3fdba35f04,79.36,High,meaningful_visibility,Manually inspect and decide whether to refresh...,515,0,0.00,21.1,165,104,0.00,23.26
4,content_b005f46f2e4c,client_3fdba35f04,79.28,High,meaningful_visibility|low_ctr_visible_page|low...,"Review title, metadata, intent match, and sear...",673,1,0.15,2.7,165,104,0.00,23.21
5,content_ff8019ba021d,client_3fdba35f04,79.27,High,meaningful_visibility,Manually inspect and decide whether to refresh...,894,1,0.11,33.7,155,104,9.09,35.71
6,content_6cb1ccf2085a,client_3fdba35f04,79.11,High,model_review_signal,Manually inspect and decide whether to refresh...,417,0,0.00,15.8,223,104,0.00,34.62
7,content_eb0ce218868b,client_3fdba35f04,79.08,High,meaningful_visibility|low_engagement,"Review structure, readability, scanability, an...",3495,5,0.14,27.0,284,104,2.63,23.53
8,content_03475a5923fe,client_3fdba35f04,79.06,High,meaningful_visibility|low_ctr_visible_page,"Review title, metadata, intent match, and sear...",2419,3,0.12,6.2,139,104,0.00,36.84
9,content_fac9e8c13f4c,client_3fdba35f04,79.00,High,meaningful_visibility|low_ctr_visible_page,"Review title, metadata, intent match, and sear...",858,0,0.00,17.4,165,104,0.00,27.27



Priority distribution:
priority_band
Medium     15183
Monitor    12643
High        2174
Name: count, dtype: int64

Top reason codes:
reason_codes
model_review_signal                                                              10098
meaningful_visibility|low_ctr_visible_page                                        4297
meaningful_visibility                                                             3294
page_one_decay_risk                                                               2897
meaningful_visibility|low_engagement                                              2893
meaningful_visibility|low_ctr_visible_page|low_engagement                         2068
meaningful_visibility|low_ctr_visible_page|page_one_decay_risk                    2021
meaningful_visibility|low_ctr_visible_page|page_one_decay_risk|low_engagement     1345
meaningful_visibility|page_one_decay_risk|low_engagement                           545
low_engagement                                                        

# 2. Intended Use and Limits

This ranked queue is intended as a decision-support tool for content reviewers. Pages with higher priority scores should be reviewed first, but the model does not automatically decide which pages must be refreshed. The recommendations are based on observed search and engagement signals from the starter dataset and may not generalize to all websites or future time periods.

# 3. Human Review + No-Go List

Every recommendation should be reviewed by a human before any content changes are made.

## Human review checklist

- Verify that the page still matches user search intent.
- Check whether the information is outdated.
- Review title, headings, metadata, and content quality.
- Compare against competing search results.
- Decide whether to refresh, monitor, merge, or leave unchanged.

## What should NOT be automated

- Publishing content updates.
- Deleting pages.
- Changing metadata without review.
- Making SEO decisions based only on model predictions.
- Assuming the model explains why rankings changed.

In [9]:
import pandas as pd
import os

os.makedirs("work/outputs", exist_ok=True)

# Names of original + missing-indicator features created by the imputer
transformed_feature_names = (
    queue_model.named_steps["imputer"]
    .get_feature_names_out(features)
)

feature_importance = pd.DataFrame({
    "Feature": transformed_feature_names,
    "Importance": queue_model.named_steps["model"].feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance.to_csv(
    "work/outputs/feature_importance.csv",
    index=False
)

display(feature_importance.head(15))
print("Feature importance exported successfully.")

,Feature,Importance
5,impressions_90d,0.268496
11,avg_position,0.171157
8,content_age_days,0.134270
4,char_count,0.045663
6,clicks_90d,0.045226
3,word_count,0.039038
10,ctr,0.037720
9,days_since_last_update,0.035529
13,scroll_rate,0.034399
7,sessions_90d,0.031499


Feature importance exported successfully.


# 4. Monitoring / Retrain Triggers

The model should be monitored periodically to ensure that its recommendations remain useful.

Possible retraining triggers include:

- Large changes in search traffic patterns.
- Significant decreases in model performance.
- Availability of newer search performance data.
- Changes in feature distributions.
- Addition of new clients or content categories.